In [ ]:
!pip install torchvision
!pip install fvcore umap-learn scikit-video opencv-python-headless matplotlib seaborn tqdm

In [ ]:
# Install required dependencies
!pip install fvcore umap-learn

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel  # For DataParallel
import torch.hub  # Use torch.hub for model loading
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc, balanced_accuracy_score,
    matthews_corrcoef, silhouette_score
)
from scipy.stats import kendalltau, spearmanr, entropy, wasserstein_distance
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from collections import Counter
warnings.filterwarnings("ignore")


# Define the VideoFeatureExtractor class using I3D via torch.hub from pytorchvideo
class VideoFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super(VideoFeatureExtractor, self).__init__()
        # Load I3D model pretrained on Kinetics from pytorchvideo via torch.hub
        self.model = torch.hub.load('facebookresearch/pytorchvideo:main', 'i3d_r50', pretrained=pretrained)
        # Remove the classification head to get features
        self.model.blocks[-1] = nn.Identity()  # Replace the final block (head) with Identity

    def forward(self, x):
        # Input x: (batch_size, C, T, H, W), e.g., (batch_size, 3, 32, 224, 224)
        features = self.model(x)  # Pass x directly to the model
        if features.dim() > 2:
            features = torch.mean(features, dim=[2, 3, 4])  # Global average pooling if needed
        return features  # Output: (batch_size, feature_dim)


# Modified VideoDataset with Optical Flow Variance
class VideoDataset(Dataset):
    def __init__(self, data_dir, label_encoder, transform=None, fraction=1.0):
        self.data_dir = data_dir
        self.label_encoder = label_encoder
        self.transform = transform
        self.fraction = fraction
        self.video_files = self._get_video_files()
        self.augmentations = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
            transforms.RandomRotation(15),
            transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0)),
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3)
        ])
        self.flow_variances = self._compute_all_flow_variances()

    def _get_video_files(self):
        video_files = []
        for class_name in os.listdir(self.data_dir):
            class_path = os.path.join(self.data_dir, class_name)
            if os.path.isdir(class_path):
                files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.mp4')]
                num_samples = max(1, int(len(files) * self.fraction))
                sampled_files = random.sample(files, num_samples) if len(files) > num_samples else files
                video_files.extend([(f, class_name) for f in sampled_files])
        return video_files

    def _compute_flow_variance(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total_frames - 1, 16, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frames.append(frame)
        cap.release()
        flows = []
        for i in range(len(frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(frames[i], frames[i + 1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flows.append(mag)
        if not flows:
            return 0
        return np.mean([np.var(f) for f in flows])

    def _compute_all_flow_variances(self):
        variances = []
        for video_path, _ in tqdm(self.video_files, desc="Computing Flow Variance"):
            variances.append(self._compute_flow_variance(video_path))
        return np.array(variances)

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path, class_name = self.video_files[idx]
        frames = self._extract_frames(video_path)
        if frames is None:
            frames = torch.zeros(32, 3, 224, 224)
            label = 0
        else:
            if random.random() < 0.5:
                frames = [torch.flip(frame, [-1]) for frame in frames]
            augmented_frames = []
            for frame in frames:
                frame = self.augmentations(frame)
                if self.transform:
                    frame = self.transform(frame)
                augmented_frames.append(frame)
            frames = torch.stack(augmented_frames).permute(1, 0, 2, 3)  # Shape: (C, T, H, W)
            label = self.label_encoder.transform([class_name])[0]
        return frames, label

    def _extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total - 1, 32, dtype=int)  # 32 frames for I3D
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = torch.tensor(frame / 255.0, dtype=torch.float32).permute(2, 0, 1)
                frames.append(frame)
        cap.release()
        if len(frames) < 32:
            last_frame = frames[-1] if frames else torch.zeros(3, 224, 224)
            frames += [last_frame.clone() for _ in range(32 - len(frames))]
        return frames


def extract_features(model, loader, device='cuda'):
    model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)  # Wrap model for multi-GPU support
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for frames, lbls in tqdm(loader, desc="Extracting Features"):
            frames = frames.to(device)  # Shape: (batch_size, C, T, H, W)
            feats = model(frames)       # Shape: (batch_size, feature_dim)
            features.append(feats.cpu().numpy())
            labels.extend(lbls.numpy())
    features = np.vstack(features)
    assert features.ndim == 2, f"Expected 2D features, got shape {features.shape}"
    return features, np.array(labels)


# Hybrid Classifier Class
class HybridClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, C=1.0, gamma='scale', kernel='rbf', weight=0.5):
        self.n_neighbors = n_neighbors
        self.C = C
        self.gamma = gamma
        self.kernel = kernel
        self.weight = weight
        self.knn = None
        self.svm = None
        self.classes_ = None

    def fit(self, X, y):
        self.knn = KNeighborsClassifier(n_neighbors=self.n_neighbors).fit(X, y)
        self.svm = SVC(C=self.C, gamma=self.gamma, kernel=self.kernel, probability=True).fit(X, y)
        self.classes_ = self.svm.classes_
        return self

    def predict(self, X):
        knn_dists, knn_indices = self.knn.kneighbors(X)
        neighbor_labels = self.knn._y[knn_indices]
        label_counts = np.apply_along_axis(lambda x: np.bincount(x, minlength=len(self.classes_)), 1, neighbor_labels)
        knn_contrib = label_counts / self.n_neighbors
        svm_probs = self.svm.predict_proba(X)
        hybrid_probs = (1 - self.weight) * svm_probs + self.weight * knn_contrib
        return np.argmax(hybrid_probs, axis=1)

    def predict_proba(self, X):
        knn_dists, knn_indices = self.knn.kneighbors(X)
        neighbor_labels = self.knn._y[knn_indices]
        label_counts = np.apply_along_axis(lambda x: np.bincount(x, minlength=len(self.classes_)), 1, neighbor_labels)
        knn_contrib = label_counts / self.n_neighbors
        svm_probs = self.svm.predict_proba(X)
        hybrid_probs = (1 - self.weight) * svm_probs + self.weight * knn_contrib
        return hybrid_probs


# Fréchet Video Distance
def calculate_fvd(real_features, fake_features):
    mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = np.mean(fake_features, axis=0), np.cov(fake_features, rowvar=False)
    diff = np.sum((mu1 - mu2)**2)
    covmean = np.sqrt(np.linalg.det(sigma1 @ sigma2))
    fid = diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid


# Evaluation Function with all metrics
def evaluate(true_labels, pred_labels, label_encoder, train_features, test_features, flow_variances, feature_importance=None):
    metrics = {
        'F1': f1_score(true_labels, pred_labels, average='weighted'),
        'Precision': precision_score(true_labels, pred_labels, average='weighted'),
        'Recall': recall_score(true_labels, pred_labels, average='weighted'),
        'Accuracy': accuracy_score(true_labels, pred_labels),
        'Kendall': kendalltau(true_labels, pred_labels)[0],
        'Spearman': spearmanr(true_labels, pred_labels)[0],
        'Balanced Accuracy': balanced_accuracy_score(true_labels, pred_labels),
        'MCC': matthews_corrcoef(true_labels, pred_labels),
        'Silhouette Score': silhouette_score(test_features, true_labels, metric='euclidean') if len(np.unique(true_labels)) > 1 else 0
    }
    print("\nEvaluation Results:")
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(14, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    plt.show()

    # Per-Class F1 Score
    per_class_f1 = f1_score(true_labels, pred_labels, average=None)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=label_encoder.classes_, y=per_class_f1)
    plt.xticks(rotation=45)
    plt.title('Per-Class F1 Score')
    plt.ylabel('F1 Score')
    plt.xlabel('Class')
    plt.tight_layout()
    plt.savefig('per_class_f1.png')
    plt.show()

    # ROC-AUC (if binary classification)
    if len(label_encoder.classes_) == 2:
        scores = hybrid_model.predict_proba(test_features)[:, 1]
        fpr, tpr, _ = roc_curve(true_labels, scores)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc="lower right")
        plt.savefig('roc_curve.png')
        plt.show()

    # Fréchet Video Distance
    fvd = calculate_fvd(train_features, test_features)
    print(f"\nFréchet Video Distance: {fvd:.4f}")

    # Wasserstein Distance
    wasserstein_dist = 0
    for i in range(train_features.shape[1]):
        wasserstein_dist += wasserstein_distance(train_features[:, i], test_features[:, i])
    wasserstein_dist /= train_features.shape[1]
    print(f"Wasserstein Distance: {wasserstein_dist:.4f}")

    # Optical Flow Variance Visualization
    plt.figure(figsize=(10, 6))
    sns.histplot(flow_variances['train'], kde=True, label='Train', alpha=0.5)
    sns.histplot(flow_variances['test'], kde=True, label='Test', alpha=0.5)
    plt.title('Optical Flow Variance Distribution')
    plt.xlabel('Variance')
    plt.ylabel('Density')
    plt.legend()
    plt.savefig('optical_flow_variance.png')
    plt.show()

    # Feature Importance (PCA-based approximation)
    if feature_importance is not None:
        plt.figure(figsize=(12, 6))
        sns.barplot(x=np.arange(feature_importance.shape[0]), y=feature_importance)
        plt.title('Feature Importance (PCA-based)')
        plt.xlabel('Principal Components')
        plt.ylabel('Importance')
        plt.savefig('feature_importance.png')
        plt.show()

    # t-SNE & UMAP Visualization
    combined_features = np.vstack([train_features, test_features])
    dataset_labels = ['Train'] * len(train_features) + ['Test'] * len(test_features)

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(combined_features)//2))
    reduced = tsne.fit_transform(combined_features)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=dataset_labels, alpha=0.7, palette='Set1', s=50)
    plt.title('t-SNE Visualization: Train vs Test Feature Space')
    plt.savefig('tsne_dataset_comparison.png')
    plt.show()

    reducer = umap.UMAP(random_state=42)
    reduced_umap = reducer.fit_transform(combined_features)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=reduced_umap[:, 0], y=reduced_umap[:, 1], hue=dataset_labels, alpha=0.7, palette='Set2', s=50)
    plt.title('UMAP Visualization: Train vs Test Feature Space')
    plt.savefig('umap_dataset_comparison.png')
    plt.show()

    # Jensen-Shannon Divergence between class distributions
    train_counter = Counter(train_dataset.labels)
    test_counter = Counter(test_dataset.labels)
    all_classes = label_encoder.classes_
    train_dist = np.array([train_counter.get(i, 0) for i in range(len(all_classes))])
    test_dist = np.array([test_counter.get(i, 0) for i in range(len(all_classes))])
    train_dist = train_dist / train_dist.sum()
    test_dist = test_dist / test_dist.sum()
    m = 0.5 * (train_dist + test_dist)
    js_div = 0.5 * (entropy(train_dist, m) + entropy(test_dist, m))
    print(f"\nJensen-Shannon Divergence between train and test label distributions: {js_div:.4f}")


# Grid Search Helper
def build_grid(best_params):
    grid = {}
    best_n = best_params['classifier__n_neighbors']
    grid['classifier__n_neighbors'] = [max(3, best_n - 2), best_n, best_n + 2]
    best_C = best_params['classifier__C']
    grid['classifier__C'] = [best_C / 2, best_C, best_C * 2]
    best_gamma = best_params['classifier__gamma']
    if isinstance(best_gamma, str):
        grid['classifier__gamma'] = [best_gamma, 'auto']
    else:
        grid['classifier__gamma'] = [best_gamma * 0.5, best_gamma, best_gamma * 2]
    best_kernel = best_params['classifier__kernel']
    grid['classifier__kernel'] = [best_kernel, 'linear' if best_kernel != 'linear' else 'rbf']
    best_weight = best_params['classifier__weight']
    grid['classifier__weight'] = [max(0.0, best_weight - 0.1), best_weight, min(1.0, best_weight + 0.1)]
    return grid


# Main Function
def main():
    global train_dataset, test_dataset, hybrid_model
    train_dir = '/kaggle/input/sum-yt/Youtube summary/Train'
    test_dir = '/kaggle/input/sum-yt/Youtube summary/Test'

    label_encoder = LabelEncoder()
    class_dirs = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    label_encoder.fit(class_dirs)

    transform = transforms.Compose([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = VideoDataset(train_dir, label_encoder, transform=transform, fraction=1.0)
    test_dataset = VideoDataset(test_dir, label_encoder, transform=transform, fraction=1.0)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=6, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=6, pin_memory=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = VideoFeatureExtractor(pretrained=True).to(device)

    print("Extracting training features...")
    train_features, train_labels = extract_features(model, train_loader, device)
    print("Extracting testing features...")
    test_features, test_labels = extract_features(model, test_loader, device)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', HybridClassifier())
    ])

    param_dist_random = {
        'classifier__n_neighbors': list(range(3, 51, 2)),
        'classifier__C': np.logspace(-3, 4, 8),
        'classifier__gamma': ['scale', 'auto'] + list(np.logspace(-4, 2, 7)),
        'classifier__kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
        'classifier__weight': np.linspace(0.0, 1.0, 11)
    }

    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist_random,
        n_iter=10,
        cv=5,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    random_search.fit(train_features, train_labels)

    print("\nBest Random Search Params:")
    for key, value in random_search.best_params_.items():
        print(f"{key}: {value}")

    param_grid = build_grid(random_search.best_params_)
    grid_search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(train_features, train_labels)

    print("\nBest Grid Search Params:")
    for key, value in grid_search.best_params_.items():
        print(f"{key}: {value}")

    hybrid_model = grid_search.best_estimator_
    predictions = hybrid_model.predict(test_features)

    # Compute PCA-based feature importance
    pca = PCA(n_components=min(train_features.shape[1], 10))
    pca.fit(train_features)
    feature_importance = np.abs(pca.components_).mean(axis=0)

    # Collect flow variances
    flow_variances = {
        'train': train_dataset.flow_variances,
        'test': test_dataset.flow_variances
    }

    # Evaluate with all metrics
    evaluate(test_labels, predictions, label_encoder, train_features, test_features, flow_variances, feature_importance)


if __name__ == "__main__":
    main()